# Phase 4 — Two-Stage Length of Stay (LOS) Prediction (Hospital & ICU)

## Technical & Methodological Rationale

Multiple MIMIC-IV LOS studies found that direct regression using only early/admission-time features performs poorly across the full LOS range, because LOS has a long right tail (a small number of very long stays) that early features can't predict well. The consistently recommended framework across this literature is: (1) classify short vs. long stay first, (2) only apply regression to predict exact duration within the "short" bucket, and explicitly acknowledge the limitation that this framework is not designed to precisely predict long-stay durations.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path.cwd()
while not (project_root / 'configs' / 'config.yaml').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.utils.config import CFG
from src.utils.logger import get_logger
from src.models.los import LengthOfStayModelPipeline
from src.models.evaluation import (
    evaluate_binary_predictions,
    evaluate_regression_predictions,
    export_los_two_stage_markdown,
    find_optimal_threshold,
)
from src.visualization.model_plots import (
    plot_roc_pr_curves,
    plot_calibration_curves,
    generate_shap_plots,
)

log = get_logger('notebook_10_los')

[07/22/26 15:08:52] INFO     Config loaded from /Users/apple/Desktop/Clinical Digital Twin/configs/config.yaml     
                             (project root: /Users/apple/Desktop/Clinical Digital Twin)

## 1. Empirical Threshold Selection (Training Split 75th Percentile)

We compute the 75th percentile threshold on the training split to separate Short vs. Long stays dynamically.

In [2]:
# Initialize pipelines
pipeline_hosp = LengthOfStayModelPipeline(target_name='hospital_los')
pipeline_icu = LengthOfStayModelPipeline(target_name='icu_los')

(X_tr_h, X_val_h, X_te_h,
 y_tr_cls_h, y_val_cls_h, y_te_cls_h,
 y_tr_reg_h, y_val_reg_h, y_te_reg_h,
 sub_tr_h, sub_val_h, sub_te_h,
 hosp_p75, feat_h, df_h) = pipeline_hosp.prepare_datasets()

(X_tr_i, X_val_i, X_te_i,
 y_tr_cls_i, y_val_cls_i, y_te_cls_i,
 y_tr_reg_i, y_val_reg_i, y_te_reg_i,
 sub_tr_i, sub_val_i, sub_te_i,
 icu_p75, feat_i, df_i) = pipeline_icu.prepare_datasets()

print(f'★ Empirical 75th Percentile Hospital LOS Threshold: {hosp_p75:.2f} days')
print(f'★ Empirical 75th Percentile ICU LOS Threshold     : {icu_p75:.2f} days')

[07/22/26 15:08:57] INFO     Preparing dataset for target 'hospital_los' (los_days)...

[07/22/26 15:09:16] INFO     Computing pre-admission prior utilization & Charlson index features dynamically...

                    INFO     Reading (full) admissions.csv

[07/22/26 15:09:20] INFO     Loaded admissions.csv: 546028 rows × 16 cols (305.5 MB)

[07/22/26 15:09:22] INFO      rows=546028, cols=16, dup_rows=0, mem=30.4 MB, overall_missing=11.9%

...

                    INFO     Dropped columns (154): ['cci_cerebrovascular_disease', 'cci_congestive_heart_failure',
                             'cci_connective_tissue_disease', 'cci_copd', 'cci_dementia',                          
                             'cci_diabetes_complicated', 'cci_diabetes_uncomplicated', 'cci_hemiplegia_paraplegia',
                             'cci_malignancy', 'cci_metastatic_tumor', 'cci_mild_liver_disease',                   
                             'cci_myocardial_infarction', 'cci_peptic_ulcer', 'cci_peripheral_vascular_disease',   
                             'cci_severe_liver_disease']

                    INFO       ... and 139 more columns

                    INFO     Selected 110 tabular predictor features for model training

                    INFO     Prevalence of Long Stay (>4.18 days) by Split:                                        
                               Train: 14939 / 59756 (25.00%)                                                       
                               Val  : 3052 / 12597 (24.23%)                                                        
                               Test : 3216 / 12876 (24.98%)

★ Empirical 75th Percentile Hospital LOS Threshold: 5.63 days
★ Empirical 75th Percentile ICU LOS Threshold     : 4.18 days


## 2. Hospital Length of Stay (`los_days`) — Two-Stage Model Training & Evaluation

In [3]:
# Stage A Classification (Hospital LOS)
logreg_a_h, tr_p_lr_h, val_p_lr_h, test_p_lr_h = pipeline_hosp.train_stageA_logistic_regression(X_tr_h, y_tr_cls_h, X_val_h, X_te_h)
xgb_a_h, tr_p_xgb_h, val_p_xgb_h, test_p_xgb_h = pipeline_hosp.train_stageA_xgboost(X_tr_h, y_tr_cls_h, sub_tr_h, X_val_h, X_te_h)
lgb_a_h, tr_p_lgb_h, val_p_lgb_h, test_p_lgb_h = pipeline_hosp.train_stageA_lightgbm(X_tr_h, y_tr_cls_h, sub_tr_h, X_val_h, X_te_h)

t_lr_h = find_optimal_threshold(y_val_cls_h, val_p_lr_h, target_recall=0.80)
t_xgb_h = find_optimal_threshold(y_val_cls_h, val_p_xgb_h, target_recall=0.80)
t_lgb_h = find_optimal_threshold(y_val_cls_h, val_p_lgb_h, target_recall=0.80)

res_lr_h = evaluate_binary_predictions(y_te_cls_h, test_p_lr_h, threshold=t_lr_h, model_name='Logistic Regression', run_name='Stage A')
res_xgb_h = evaluate_binary_predictions(y_te_cls_h, test_p_xgb_h, threshold=t_xgb_h, model_name='XGBoost', run_name='Stage A')
res_lgb_h = evaluate_binary_predictions(y_te_cls_h, test_p_lgb_h, threshold=t_lgb_h, model_name='LightGBM', run_name='Stage A')

# Isotonic Calibration
iso_h, val_p_cal_h, test_p_cal_h = pipeline_hosp.calibrate_stageA_predictions(y_val_cls_h, val_p_lgb_h, test_p_lgb_h)
t_cal_h = find_optimal_threshold(y_val_cls_h, val_p_cal_h, target_recall=0.80)
res_cal_h = evaluate_binary_predictions(y_te_cls_h, test_p_cal_h, threshold=t_cal_h, model_name='LightGBM (Calibrated)', run_name='Stage A')

res_lr_h['target_name'] = 'Hospital LOS'
res_xgb_h['target_name'] = 'Hospital LOS'
res_lgb_h['target_name'] = 'Hospital LOS'
res_cal_h['target_name'] = 'Hospital LOS'

# Stage B Regression (Hospital LOS)
tr_short_m_h = (y_tr_reg_h <= hosp_p75)
val_short_m_h = (y_val_reg_h <= hosp_p75)
te_act_short_m_h = (y_te_reg_h <= hosp_p75)

xgb_b_h, _, te_pred_xgb_b_h = pipeline_hosp.train_stageB_xgboost(X_tr_h.iloc[tr_short_m_h], y_tr_reg_h[tr_short_m_h], sub_tr_h[tr_short_m_h], X_val_h.iloc[val_short_m_h], X_te_h)
lgb_b_h, _, te_pred_lgb_b_h = pipeline_hosp.train_stageB_lightgbm(X_tr_h.iloc[tr_short_m_h], y_tr_reg_h[tr_short_m_h], sub_tr_h[tr_short_m_h], X_val_h.iloc[val_short_m_h], X_te_h)

# Deployment predicted short bucket
pred_long_h = (test_p_cal_h >= t_cal_h).astype(int)
te_pred_short_m_h = (pred_long_h == 0)

res_reg_lgb_dep_h = evaluate_regression_predictions(y_te_reg_h[te_pred_short_m_h], te_pred_lgb_b_h[te_pred_short_m_h], 'LightGBM Regressor', 'Predicted Short Bucket (Deployment Primary)', 'Hospital LOS')
res_reg_xgb_dep_h = evaluate_regression_predictions(y_te_reg_h[te_pred_short_m_h], te_pred_xgb_b_h[te_pred_short_m_h], 'XGBoost Regressor', 'Predicted Short Bucket (Deployment Primary)', 'Hospital LOS')
res_reg_lgb_opt_h = evaluate_regression_predictions(y_te_reg_h[te_act_short_m_h], te_pred_lgb_b_h[te_act_short_m_h], 'LightGBM Regressor', 'Actual Short Bucket (Optimistic Upper Bound)', 'Hospital LOS')
res_reg_xgb_opt_h = evaluate_regression_predictions(y_te_reg_h[te_act_short_m_h], te_pred_xgb_b_h[te_act_short_m_h], 'XGBoost Regressor', 'Actual Short Bucket (Optimistic Upper Bound)', 'Hospital LOS')

[07/22/26 15:10:39] INFO     Stage A: Training Logistic Regression (L2, class_weight='balanced')...

[07/22/26 15:10:49] INFO     Stage A: Training XGBoost with GroupKFold search & scale_pos_weight...

[07/22/26 15:11:37] INFO     Best Stage A XGBoost hyperparams: {'max_depth': 5, 'learning_rate': 0.05,             
                             'n_estimators': 300} (CV accuracy: 0.6684)

[07/22/26 15:11:47] INFO     Stage A: Training LightGBM with GroupKFold search & class_weight='balanced'...

[07/22/26 15:12:19] INFO     Best Stage A LightGBM hyperparams: {'num_leaves': 63, 'learning_rate': 0.03,          
                             'n_estimators': 350} (CV accuracy: 0.6760)

...

[07/22/26 15:12:41] INFO     Stage B: Training LightGBM Regressor on short stay bucket (N=286056)...

[07/22/26 15:12:45] INFO     [LightGBM Regressor - Hospital LOS] (Predicted Short Bucket (Deployment Primary),     
                             N=44580) MAE: 1.3622 days | RMSE: 3.3330 days | R²: 0.1162

                    INFO     [XGBoost Regressor - Hospital LOS] (Predicted Short Bucket (Deployment Primary),      
                             N=44580) MAE: 1.3640 days | RMSE: 3.3309 days | R²: 0.1172

                    INFO     [LightGBM Regressor - Hospital LOS] (Actual Short Bucket (Optimistic Upper Bound),    
                             N=62221) MAE: 0.8723 days | RMSE: 1.1196 days | R²: 0.4401

                    INFO     [XGBoost Regressor - Hospital LOS] (Actual Short Bucket (Optimistic Upper Bound),     
                             N=62221) MAE: 0.8745 days | RMSE: 1.1216 days | R²: 0.4380

## 3. ICU Length of Stay (`icu_los_days`) — Two-Stage Model Training & Evaluation

In [4]:
# Stage A Classification (ICU LOS)
logreg_a_i, tr_p_lr_i, val_p_lr_i, test_p_lr_i = pipeline_icu.train_stageA_logistic_regression(X_tr_i, y_tr_cls_i, X_val_i, X_te_i)
xgb_a_i, tr_p_xgb_i, val_p_xgb_i, test_p_xgb_i = pipeline_icu.train_stageA_xgboost(X_tr_i, y_tr_cls_i, sub_tr_i, X_val_i, X_te_i)
lgb_a_i, tr_p_lgb_i, val_p_lgb_i, test_p_lgb_i = pipeline_icu.train_stageA_lightgbm(X_tr_i, y_tr_cls_i, sub_tr_i, X_val_i, X_te_i)

t_lr_i = find_optimal_threshold(y_val_cls_i, val_p_lr_i, target_recall=0.80)
t_xgb_i = find_optimal_threshold(y_val_cls_i, val_p_xgb_i, target_recall=0.80)
t_lgb_i = find_optimal_threshold(y_val_cls_i, val_p_lgb_i, target_recall=0.80)

res_lr_i = evaluate_binary_predictions(y_te_cls_i, test_p_lr_i, threshold=t_lr_i, model_name='Logistic Regression', run_name='Stage A')
res_xgb_i = evaluate_binary_predictions(y_te_cls_i, test_p_xgb_i, threshold=t_xgb_i, model_name='XGBoost', run_name='Stage A')
res_lgb_i = evaluate_binary_predictions(y_te_cls_i, test_p_lgb_i, threshold=t_lgb_i, model_name='LightGBM', run_name='Stage A')

# Isotonic Calibration
iso_i, val_p_cal_i, test_p_cal_i = pipeline_icu.calibrate_stageA_predictions(y_val_cls_i, val_p_lgb_i, test_p_lgb_i)
t_cal_i = find_optimal_threshold(y_val_cls_i, val_p_cal_i, target_recall=0.80)
res_cal_i = evaluate_binary_predictions(y_te_cls_i, test_p_cal_i, threshold=t_cal_i, model_name='LightGBM (Calibrated)', run_name='Stage A')

res_lr_i['target_name'] = 'ICU LOS'
res_xgb_i['target_name'] = 'ICU LOS'
res_lgb_i['target_name'] = 'ICU LOS'
res_cal_i['target_name'] = 'ICU LOS'

# Stage B Regression (ICU LOS)
tr_short_m_i = (y_tr_reg_i <= icu_p75)
val_short_m_i = (y_val_reg_i <= icu_p75)
te_act_short_m_i = (y_te_reg_i <= icu_p75)

xgb_b_i, _, te_pred_xgb_b_i = pipeline_icu.train_stageB_xgboost(X_tr_i.iloc[tr_short_m_i], y_tr_reg_i[tr_short_m_i], sub_tr_i[tr_short_m_i], X_val_i.iloc[val_short_m_i], X_te_i)
lgb_b_i, _, te_pred_lgb_b_i = pipeline_icu.train_stageB_lightgbm(X_tr_i.iloc[tr_short_m_i], y_tr_reg_i[tr_short_m_i], sub_tr_i[tr_short_m_i], X_val_i.iloc[val_short_m_i], X_te_i)

# Deployment predicted short bucket
pred_long_i = (test_p_cal_i >= t_cal_i).astype(int)
te_pred_short_m_i = (pred_long_i == 0)

res_reg_lgb_dep_i = evaluate_regression_predictions(y_te_reg_i[te_pred_short_m_i], te_pred_lgb_b_i[te_pred_short_m_i], 'LightGBM Regressor', 'Predicted Short Bucket (Deployment Primary)', 'ICU LOS')
res_reg_xgb_dep_i = evaluate_regression_predictions(y_te_reg_i[te_pred_short_m_i], te_pred_xgb_b_i[te_pred_short_m_i], 'XGBoost Regressor', 'Predicted Short Bucket (Deployment Primary)', 'ICU LOS')
res_reg_lgb_opt_i = evaluate_regression_predictions(y_te_reg_i[te_act_short_m_i], te_pred_lgb_b_i[te_act_short_m_i], 'LightGBM Regressor', 'Actual Short Bucket (Optimistic Upper Bound)', 'ICU LOS')
res_reg_xgb_opt_i = evaluate_regression_predictions(y_te_reg_i[te_act_short_m_i], te_pred_xgb_b_i[te_act_short_m_i], 'XGBoost Regressor', 'Actual Short Bucket (Optimistic Upper Bound)', 'ICU LOS')

                    INFO     Stage A: Training Logistic Regression (L2, class_weight='balanced')...

[07/22/26 15:12:48] INFO     Stage A: Training XGBoost with GroupKFold search & scale_pos_weight...

[07/22/26 15:13:01] INFO     Best Stage A XGBoost hyperparams: {'max_depth': 6, 'learning_rate': 0.03,             
                             'n_estimators': 350} (CV accuracy: 0.6218)

[07/22/26 15:13:04] INFO     Stage A: Training LightGBM with GroupKFold search & class_weight='balanced'...

[07/22/26 15:13:13] INFO     Best Stage A LightGBM hyperparams: {'num_leaves': 63, 'learning_rate': 0.03,          
                             'n_estimators': 350} (CV accuracy: 0.6256)

...

[07/22/26 15:13:18] INFO     Stage B: Training LightGBM Regressor on short stay bucket (N=44817)...

[07/22/26 15:13:19] INFO     [LightGBM Regressor - ICU LOS] (Predicted Short Bucket (Deployment Primary), N=3812)  
                             MAE: 1.7935 days | RMSE: 4.5315 days | R²: -0.0645

                    INFO     [XGBoost Regressor - ICU LOS] (Predicted Short Bucket (Deployment Primary), N=3812)   
                             MAE: 1.7927 days | RMSE: 4.5318 days | R²: -0.0647

                    INFO     [LightGBM Regressor - ICU LOS] (Actual Short Bucket (Optimistic Upper Bound), N=9660) 
                             MAE: 0.7855 days | RMSE: 0.9568 days | R²: 0.0327

                    INFO     [XGBoost Regressor - ICU LOS] (Actual Short Bucket (Optimistic Upper Bound), N=9660)  
                             MAE: 0.7851 days | RMSE: 0.9562 days | R²: 0.0339

## 4. Model Artifact Saving & Markdown Export

In [5]:
pipeline_hosp.save_artifacts(xgb_a_h, lgb_a_h, iso_h, xgb_b_h, lgb_b_h)
pipeline_icu.save_artifacts(xgb_a_i, lgb_a_i, iso_i, xgb_b_i, lgb_b_i)

all_cls = [res_lr_h, res_xgb_h, res_lgb_h, res_cal_h, res_lr_i, res_xgb_i, res_lgb_i, res_cal_i]
all_reg = [res_reg_lgb_dep_h, res_reg_xgb_dep_h, res_reg_lgb_opt_h, res_reg_xgb_opt_h, res_reg_lgb_dep_i, res_reg_xgb_dep_i, res_reg_lgb_opt_i, res_reg_xgb_opt_i]

report_file = export_los_two_stage_markdown(all_cls, all_reg, hosp_p75, icu_p75)
print(f'Successfully exported report → {report_file}')

                    INFO     Saved LOS model pickle → models/los_stageA_classifier_xgboost.pkl

                    INFO     Saved LOS model pickle → models/los_stageA_classifier_lightgbm.pkl

                    INFO     Saved LOS model pickle → models/los_stageA_calibrated.pkl

                    INFO     Saved LOS model pickle → models/los_stageB_regressor_xgboost.pkl

                    INFO     Saved LOS model pickle → models/los_stageB_regressor_lightgbm.pkl

...

                    INFO     Saved ICU_LOS model pickle → models/icu_los_stageA_calibrated.pkl

                    INFO     Saved ICU_LOS model pickle → models/icu_los_stageB_regressor_xgboost.pkl

                    INFO     Saved ICU_LOS model pickle → models/icu_los_stageB_regressor_lightgbm.pkl

[07/22/26 15:13:20] INFO     Saved two-stage LOS comparison report → /Users/apple/Desktop/Clinical Digital         
                             Twin/reports/tables/los_two_stage_results.md

Successfully exported report → /Users/apple/Desktop/Clinical Digital Twin/reports/tables/los_two_stage_results.md


## 5. SHAP Explainability & Visualizations

In [6]:
fig_dir = Path(CFG.resolve(CFG.paths.reports)) / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

generate_shap_plots(lgb_a_h, X_tr_h, X_te_h, y_te_cls_h, test_p_lgb_h, t_lgb_h, output_dir=fig_dir, output_prefix='los_stageA')
generate_shap_plots(lgb_a_i, X_tr_i, X_te_i, y_te_cls_i, test_p_lgb_i, t_lgb_i, output_dir=fig_dir, output_prefix='icu_los_stageA')

                    INFO     Computing SHAP values for model explainability...

[07/22/26 15:13:24] INFO     TOP-10 SHAP FEATURES FOR MODEL:

                    INFO        1. admission_type_EU OBSERVATION       (mean |SHAP| = 1.3618)

                    INFO        2. admission_type_DIRECT OBSERVATION   (mean |SHAP| = 0.3157)

                    INFO        3. prior_cumulative_los_days           (mean |SHAP| = 0.2629)

...

[07/22/26 15:13:35] INFO     Saved SHAP summary plot → /Users/apple/Desktop/Clinical Digital                       
                             Twin/reports/figures/icu_los_stageA_shap_summary.png

                    INFO     Saved individual SHAP plot (tp) → /Users/apple/Desktop/Clinical Digital               
                             Twin/reports/figures/mortality_shap_tp.png

[07/22/26 15:13:36] INFO     Saved individual SHAP plot (tn) → /Users/apple/Desktop/Clinical Digital               
                             Twin/reports/figures/mortality_shap_tn.png

[07/22/26 15:13:37] INFO     Saved individual SHAP plot (fp) → /Users/apple/Desktop/Clinical Digital               
                             Twin/reports/figures/mortality_shap_fp.png

{'summary': PosixPath('/Users/apple/Desktop/Clinical Digital Twin/reports/figures/icu_los_stageA_shap_summary.png'),
 'tp': PosixPath('/Users/apple/Desktop/Clinical Digital Twin/reports/figures/mortality_shap_tp.png'),
 'tn': PosixPath('/Users/apple/Desktop/Clinical Digital Twin/reports/figures/mortality_shap_tn.png'),
 'fp': PosixPath('/Users/apple/Desktop/Clinical Digital Twin/reports/figures/mortality_shap_fp.png')}